# Split sentence files into per-episode CSV files (non-natives and translations)

Each corpus has a `.tok` file (English sentences) and a `.dat` file (metadata) that are **line-aligned**:
line *n* of the `.tok` is described by line *n* of the `.dat`.

Everything is driven by the sentences in your sentence files (`NN_France.txt`, `T_France`, ...). For every sentence file this notebook:

1. searches each sentence in the corpus `.tok` file,
2. takes the metadata of that exact line from the corpus `.dat` file,
3. groups the matched sentences by episode (`SESSION_ID`) and reports exactly how many sentences went into each episode,
4. writes one CSV per episode (e.g. `ep-04-03-10.csv`) into `output/<corpus>/<sentence file>/`,
5. counts the speakers: in how many episodes each speaker appears, how many turns and how many sentences they have, per file and overall.

`source_language` comes **only from the sentence file** a sentence is in (NN_France / T_France -> `fr`, ...).

**How to use:** enter your files in section 1 (or answer the questions in section 2), then run all cells.

## 1. ENTER YOUR FILES HERE

File names are looked up in `INPUT_DIR`; you can also give full paths. Upper/lower case does not matter, and a missing `.txt` is tolerated.
Each sentence file gets the language of its speakers, which is written to the `source_language` column.

In [14]:
from pathlib import Path

INPUT_DIR = Path("/content")
OUTPUT_DIR = Path("/content/output")

CORPORA = {
    "nonnatives": {
        "tok": "nonnatives.tok",
        "dat": "nonnatives.dat",
        "sentence_files": {
            "NN_France.txt": "fr",
            "NN_Germany.txt": "de",
            "NN_Italy.txt": "it",
            "NN_Spain.txt": "es",
        },
    },
}

ASK_FOR_PATHS = False

ROW_MODE = "speaker"
CSV_SEP = ";"
CSV_ENCODING = "utf-8-sig"

## 2. Enter the files (only if `ASK_FOR_PATHS = True`)

For every file you are asked for a name or path. Press **Enter** to keep the value in [brackets], or type `-` to skip a sentence file.

In [15]:
from pathlib import Path

INPUT_DIR = Path("/content")
OUTPUT_DIR = Path("/content/output")

ASK_FOR_PATHS = False

## 3. Helper functions

In [16]:
import re
import html
from collections import defaultdict

import pandas as pd

ATTR_RE = re.compile(r'(\w+)="([^"]*)"')


def read_lines(path):
    """Read a text file as a list of lines.

    Splits on '\n' only (not str.splitlines) so that the line numbers of
    .tok and .dat stay exactly aligned.
    """
    lines = Path(path).read_text(encoding="utf-8").split("\n")
    if lines and lines[-1] == "":        # file ended with a newline
        lines.pop()
    return [l.rstrip("\r") for l in lines]


def find_file(name, folder=None):
    """Find a file: full path, or name inside `folder` (default INPUT_DIR).
    Ignores upper/lower case and a missing or extra '.txt'."""
    folder = Path(folder) if folder is not None else Path(INPUT_DIR)
    p = Path(str(name))
    base = p if p.is_absolute() else folder / p
    if base.is_file():
        return base
    if p.is_file():                      # relative to the notebook folder
        return p
    low = base.name.lower()
    wanted = {low, low + ".txt"} | ({low[:-4]} if low.endswith(".txt") else set())
    if base.parent.is_dir():
        for q in base.parent.iterdir():
            if q.is_file() and q.name.lower() in wanted:
                return q
    return None


def norm(sentence):
    """Normalise a sentence for matching only (the output keeps the .tok text).

    Trims / collapses whitespace and treats the quote styles as equal:
    `` and '' (used in translations.tok) -> "    and    ` -> '
    """
    s = sentence.replace("``", '"').replace("''", '"').replace("`", "'")
    return " ".join(s.split())


def parse_dat_line(line):
    """<LINE NAME="Segni" LANGUAGE="IT" ... /> -> {'NAME': 'Segni', 'LANGUAGE': 'IT', ...}"""
    return {k: html.unescape(v) for k, v in ATTR_RE.findall(line)}


def clean_name(raw):
    """'Ari Vatanen (PPE-DE ). –' -> 'Ari Vatanen'    'Roche,' -> 'Roche'"""
    name = re.sub(r"\([^)]*\)", " ", raw)       # drop political group in brackets
    name = name.strip(" \t,.;:–—-")             # drop trailing commas, dots, dashes
    return " ".join(name.split())

## 4. Check that all entered files exist

In [17]:
check_rows = []
for corpus, cfg in CORPORA.items():
    cfg["_tok_path"] = find_file(cfg["tok"])
    cfg["_dat_path"] = find_file(cfg["dat"])
    cfg["_sentence_paths"] = {}
    entries = [("tok", cfg["tok"], "", cfg["_tok_path"]), ("dat", cfg["dat"], "", cfg["_dat_path"])]
    for name, lang in cfg["sentence_files"].items():
        path = find_file(name)
        cfg["_sentence_paths"][name] = path
        entries.append(("sentences", name, lang, path))
    for role, name, lang, path in entries:
        check_rows.append({
            "corpus": corpus, "file": role, "language": lang, "entered": name,
            "found": "yes" if path else "NO",
            "path": str(path) if path else "",
            "size_MB": round(path.stat().st_size / 1e6, 1) if path else None,
        })

display(pd.DataFrame(check_rows))
for corpus, cfg in CORPORA.items():
    if not (cfg["_tok_path"] and cfg["_dat_path"]):
        print(f"{corpus}: .tok or .dat not found -> this corpus will be skipped")
    missing = [n for n, p in cfg["_sentence_paths"].items() if p is None]
    if missing:
        print(f"{corpus}: sentence files not found (skipped): {missing}")

,corpus,file,language,entered,found,path,size_MB
0,nonnatives,tok,,nonnatives.tok,yes,/content/nonnatives.tok,4.4
1,nonnatives,dat,,nonnatives.dat,yes,/content/nonnatives.dat,3.7
2,nonnatives,sentences,fr,NN_France.txt,yes,/content/NN_France.txt,0.2
3,nonnatives,sentences,de,NN_Germany.txt,yes,/content/NN_Germany.txt,0.1
4,nonnatives,sentences,it,NN_Italy.txt,yes,/content/NN_Italy.txt,0.1
5,nonnatives,sentences,es,NN_Spain.txt,yes,/content/NN_Spain.txt,0.3


## 5. Load a corpus and find the sentences in its `.tok` file

Almost every sentence occurs only once in the `.tok`, so it is simply taken there. For a sentence that occurs several times (e.g. `Mr President , I will be very brief .`)
the occurrence is chosen whose **neighbouring sentences** match the neighbours in your sentence file; if that does not decide it, the first unused occurrence after the previous match is taken.
Sentences that do not occur in the `.tok` at all are listed as `not_found`; sentences that appear in your file more often than in the `.tok` as `duplicate`.
The order of your sentence file does not need to match the `.tok` - the output is always written in `.tok` order.

In [18]:
def load_corpus(tok_path, dat_path):
    tok_lines = read_lines(tok_path)
    dat_lines = read_lines(dat_path)
    print(f"  {Path(tok_path).name}: {len(tok_lines):,} lines")
    print(f"  {Path(dat_path).name}: {len(dat_lines):,} lines")
    assert len(tok_lines) == len(dat_lines), "The .tok and .dat files do not have the same number of lines!"
    bad = [i + 1 for i, l in enumerate(dat_lines) if 'SESSION_ID="' not in l]
    assert not bad, f".dat lines without SESSION_ID: {bad[:10]}"

    tok_keys = [norm(l) for l in tok_lines]
    tok_index = defaultdict(list)            # sentence -> every line number (0-based) in the .tok
    for i, key in enumerate(tok_keys):
        tok_index[key].append(i)
    print("  files are aligned")
    return {"tok": tok_lines, "dat": dat_lines, "keys": tok_keys, "index": tok_index}


def match_sentences(sent_lines, corpus):
    """Return (matches, problems, order_jumps).

    matches     : list of (file_line_number, tok_index)   tok_index is 0-based
    problems    : list of dicts for sentences that were not found / duplicate
    order_jumps : how often the sentence file goes back to an earlier .tok position (information only)
    """
    tok_index, tok_keys = corpus["index"], corpus["keys"]
    n_tok = len(tok_keys)
    keys = [norm(l) for l in sent_lines]
    used, last, jumps = set(), -1, 0
    matches, problems = [], []

    for k, key in enumerate(keys):
        line_no = k + 1
        if not key:                                   # skip empty lines
            continue
        cands = tok_index.get(key)
        if not cands:
            problems.append({"file_line": line_no, "status": "not_found", "tok_line": None, "text": sent_lines[k]})
            continue
        free = [c for c in cands if c not in used]
        if not free:
            problems.append({"file_line": line_no, "status": "duplicate", "tok_line": cands[0] + 1, "text": sent_lines[k]})
            continue
        if len(free) > 1:
            prev_key = keys[k - 1] if k > 0 else None
            next_key = keys[k + 1] if k + 1 < len(keys) else None
            context = [c for c in free
                       if (c > 0 and tok_keys[c - 1] == prev_key)
                       or (c + 1 < n_tok and tok_keys[c + 1] == next_key)]
            if context:
                free = context
            after = [c for c in free if c > last]
            idx = after[0] if after else free[0]
        else:
            idx = free[0]
        if idx < last:
            jumps += 1
        used.add(idx)
        last = idx
        matches.append((line_no, idx))

    return matches, problems, jumps

## 6. Build the rows (line level and speaker-turn level)

`source_language` is the language of the sentence file. All other columns are copied from the matched `.dat` line
(the two corpora have different `.dat` fields, e.g. `nonnatives.dat` has `MEPID` and `STATE`, `translations.dat` does not - whatever is there is copied).

In [19]:
# .dat attribute -> column name (attributes not listed keep their name in lower case)
DAT_COLUMNS = {"SEQ_SPEAKER_ID": "speaker_id", "NAME": "speaker_name_raw", "SESSION_ID": "session_id",
               "MEPID": "mep_id", "STATE": "state", "LANGUAGE": "spoken_language"}
MAIN = ["speaker_id", "speaker_name", "source_language", "english_text"]


def build_line_table(matches, source_language, corpus):
    """One row per matched sentence, in .tok order."""
    rows = []
    for line_no, idx in sorted(matches, key=lambda m: m[1]):
        r = parse_dat_line(corpus["dat"][idx])
        row = {
            "speaker_id":      r.get("SEQ_SPEAKER_ID", ""),     # turn number within the episode
            "speaker_name":    clean_name(r.get("NAME", "")),
            "source_language": source_language,                 # from the sentence file
            "english_text":    corpus["tok"][idx].strip(),
            "session_id":      r.get("SESSION_ID", ""),
            "tok_line":        idx + 1,                         # 1-based line number in .tok and .dat
            "file_line":       line_no,                         # 1-based line number in the sentence file
        }
        for attr, value in r.items():                           # every other .dat field, as is
            col = DAT_COLUMNS.get(attr, attr.lower())
            if col not in row and col != "speaker_name_raw":
                row[col] = value
        row["speaker_name_raw"] = r.get("NAME", "")
        rows.append(row)
    return pd.DataFrame(rows)


def build_speaker_table(lines_df):
    """Join consecutive lines of the same speaker turn into one row."""
    fixed = ["english_text", "tok_line", "file_line"]
    extra = [c for c in lines_df.columns if c not in MAIN + ["session_id"] + fixed]   # other .dat fields
    key = ["session_id", "speaker_id"] + extra
    turn = (lines_df[key] != lines_df[key].shift()).any(axis=1).cumsum()
    agg = {
        "speaker_id": ("speaker_id", "first"),
        "speaker_name": ("speaker_name", "first"),
        "source_language": ("source_language", "first"),
        "english_text": ("english_text", " ".join),
        "num_lines": ("english_text", "size"),
        "file_line_start": ("file_line", "min"),
        "file_line_end": ("file_line", "max"),
        "tok_line_start": ("tok_line", "min"),
        "tok_line_end": ("tok_line", "max"),
        "session_id": ("session_id", "first"),
    }
    agg.update({c: (c, "first") for c in extra})
    return lines_df.groupby(turn, sort=False).agg(**agg).reset_index(drop=True)

## 7. Counting speakers

Names in the `.dat` files are written in different ways, so before counting they are brought together **only when it is unambiguous**:

* the political group and trailing punctuation are removed: `Ari Vatanen (PPE-DE ). –` -> `Ari Vatanen`
* `Mann, Erika` -> `Erika Mann`
* a short name is joined with a longer one only if exactly one longer name in the same file ends with it: `Vatanen` -> `Ari Vatanen`, `Wurtz` -> `Francis Wurtz`.
  `Le Pen` stays `Le Pen`, because the file has both `Jean-Marie Le Pen` and `Marine Le Pen`.
* entries naming several people (`Bordes, Cauquil und Laguiller`, joint explanations of vote) are kept as they are and marked `joint_entry = True`.
* if the `.dat` has an MEP id, it is part of the identity too (two people sharing one id stay separate).

The column `name_variants` shows every spelling that was counted as that speaker, so you can check it.

In [20]:
JOINT_RE = re.compile(r"\s(?:und|and|et)\s", re.IGNORECASE)


def is_joint(name):
    return bool(JOINT_RE.search(name)) or "," in name


def canonical_names(names):
    """Map every cleaned name of one sentence file to the name used for counting."""
    def flip(n):                                         # 'Mann, Erika' -> 'Erika Mann'
        if n.count(",") == 1 and not JOINT_RE.search(n):
            last, first = (p.strip() for p in n.split(","))
            if last and first:
                return f"{first} {last}"
        return n

    flipped = {n: flip(n) for n in set(names)}
    singles = {f for f in flipped.values() if f and not is_joint(f)}
    out = {}
    for n, f in flipped.items():
        if f in singles:
            longer = sorted((g for g in singles if g != f and g.lower().endswith(" " + f.lower())), key=len)
            # accept when there is one longer name, or they are all the same name getting longer
            chain = all(b.lower().endswith(a.lower()) for a, b in zip(longer, longer[1:]))
            out[n] = longer[-1] if longer and chain else f
        else:
            out[n] = f
    return out


def speaker_summaries(turns):
    """turns = speaker-turn table (one row per turn). Returns (speakers, speakers_by_episode)."""
    turns = turns.copy()
    mapping = canonical_names(turns["speaker_name"])
    turns["speaker"] = turns["speaker_name"].map(mapping)
    ids = turns["mep_id"] if "mep_id" in turns.columns else ""
    turns["speaker_key"] = turns["speaker"].str.lower() + "|" + ids

    speakers = (turns.groupby("speaker_key", sort=False)
                .agg(speaker=("speaker", "first"),
                     **({"mep_id": ("mep_id", "first")} if "mep_id" in turns.columns else {}),
                     source_language=("source_language", "first"),
                     num_episodes=("session_id", "nunique"),
                     num_turns=("speaker_id", "size"),
                     num_sentences=("num_lines", "sum"),
                     name_variants=("speaker_name_raw", lambda s: " | ".join(sorted(set(s)))),
                     episodes=("session_id", lambda s: ", ".join(dict.fromkeys(s))))
                .reset_index())
    speakers.insert(2, "joint_entry", speakers["speaker"].map(is_joint))
    speakers = speakers.sort_values(["num_sentences", "speaker"], ascending=[False, True]).reset_index(drop=True)

    by_episode = (turns.groupby(["speaker_key", "session_id"], sort=False)
                  .agg(speaker=("speaker", "first"),
                       num_turns=("speaker_id", "size"),
                       num_sentences=("num_lines", "sum"))
                  .reset_index())
    rank = {k: i for i, k in enumerate(speakers["speaker_key"])}
    by_episode = (by_episode.assign(_rank=by_episode["speaker_key"].map(rank))
                  .sort_values("_rank", kind="stable")
                  [["speaker", "session_id", "num_turns", "num_sentences"]]
                  .reset_index(drop=True))
    return speakers.drop(columns="speaker_key"), by_episode

## 8. Process one sentence file -> one folder with one CSV per episode

Files written into `output/<corpus>/<sentence file>/`:

* `ep-XX-XX-XX.csv` - one per episode
* `_summary_<file>.csv` - sentences per episode, with the lines of your sentence file that went into it
* `_speakers_<file>.csv` - one row per speaker: `num_episodes`, `num_turns` (times they took the floor), `num_sentences`
* `_speakers_by_episode_<file>.csv` - one row per speaker per episode
* `_unmatched_lines.csv` - only if some sentences were not found in the `.tok`

In [21]:
def save(df, path):
    df.to_csv(path, index=False, sep=CSV_SEP, encoding=CSV_ENCODING)


def process_sentence_file(path, source_language, corpus, corpus_name):
    path = Path(path)
    folder = path.stem                                   # NN_France.txt -> NN_France, T_France -> T_France
    out_dir = OUTPUT_DIR / corpus_name / folder
    out_dir.mkdir(parents=True, exist_ok=True)

    sent_lines = read_lines(path)
    n_sentences = sum(1 for l in sent_lines if norm(l))
    matches, problems, jumps = match_sentences(sent_lines, corpus)
    lines_df = build_line_table(matches, source_language, corpus)

    summary, all_turns = [], []
    if not lines_df.empty:
        for ep, ep_lines in lines_df.groupby("session_id", sort=False):
            ep_lines = ep_lines.reset_index(drop=True)
            turns = build_speaker_table(ep_lines)
            all_turns.append(turns)
            save(turns if ROW_MODE == "speaker" else ep_lines, out_dir / f"{ep}.csv")
            summary.append({
                "session_id": ep,
                "file_line_first": int(ep_lines["file_line"].min()),
                "file_line_last": int(ep_lines["file_line"].max()),
                "matched_sentences": len(ep_lines),
                "speaker_turns": len(turns),
                "speakers": ", ".join(sorted(set(ep_lines["speaker_name"]))),
                "csv_file": f"{ep}.csv",
            })
    summary_df = pd.DataFrame(summary, columns=["session_id", "file_line_first", "file_line_last",
                                                "matched_sentences", "speaker_turns", "speakers", "csv_file"])
    save(summary_df, out_dir / f"_summary_{folder}.csv")

    problems_df = pd.DataFrame(problems, columns=["file_line", "status", "tok_line", "text"])
    if not problems_df.empty:
        save(problems_df, out_dir / "_unmatched_lines.csv")

    turns_df = pd.concat(all_turns, ignore_index=True) if all_turns else pd.DataFrame()
    if not turns_df.empty:
        speakers, by_episode = speaker_summaries(turns_df)
        save(speakers, out_dir / f"_speakers_{folder}.csv")
        save(by_episode, out_dir / f"_speakers_by_episode_{folder}.csv")
    else:
        speakers, by_episode = pd.DataFrame(), pd.DataFrame()

    # ---- report ----
    n_nf = sum(p["status"] == "not_found" for p in problems)
    n_dup = sum(p["status"] == "duplicate" for p in problems)
    in_csvs = int(summary_df["matched_sentences"].sum())
    print(f"  {path.name} ({source_language})  ->  {out_dir}/")
    print(f"    sentences in file        : {n_sentences:,}")
    print(f"    matched in .tok          : {len(matches):,}")
    print(f"    not found in .tok        : {n_nf:,}")
    print(f"    duplicates in file       : {n_dup:,}")
    print(f"    sentences in episode CSVs: {in_csvs:,}  in {len(summary_df):,} episode CSV files")
    if jumps:
        print(f"    (file order differs from .tok order at {jumps:,} places - output is in .tok order)")
    if in_csvs == n_sentences and problems_df.empty:
        print("    check                    : OK - every sentence is in an episode CSV")
    else:
        print(f"    check                    : {n_sentences - in_csvs:,} sentences not placed - see _unmatched_lines.csv")
    print()

    return {"corpus": corpus_name, "folder": folder, "source_language": source_language, "out_dir": out_dir,
            "n_sentences": n_sentences, "n_not_found": n_nf, "n_duplicate": n_dup,
            "summary": summary_df, "turns": turns_df, "speakers": speakers, "speakers_by_episode": by_episode}

## 9. Run everything

In [22]:
results = []
for corpus_name, cfg in CORPORA.items():
    sentence_paths = {n: p for n, p in cfg.get("_sentence_paths", {}).items() if p}
    if not (cfg.get("_tok_path") and cfg.get("_dat_path")) or not sentence_paths:
        print(f"=== {corpus_name}: skipped (files missing, see section 4) ===\n")
        continue
    print(f"=== {corpus_name} ===")
    corpus = load_corpus(cfg["_tok_path"], cfg["_dat_path"])
    print()
    for name, path in sentence_paths.items():
        results.append(process_sentence_file(path, cfg["sentence_files"][name], corpus, corpus_name))
    del corpus                                   # free memory before the next corpus

=== nonnatives ===
  nonnatives.tok: 29,734 lines
  nonnatives.dat: 29,734 lines
  files are aligned

  NN_France.txt (fr)  ->  /content/output/nonnatives/NN_France/
    sentences in file        : 1,149
    matched in .tok          : 1,149
    not found in .tok        : 0
    duplicates in file       : 0
    sentences in episode CSVs: 1,149  in 31 episode CSV files
    check                    : OK - every sentence is in an episode CSV

  NN_Germany.txt (de)  ->  /content/output/nonnatives/NN_Germany/
    sentences in file        : 671
    matched in .tok          : 671
    not found in .tok        : 0
    duplicates in file       : 0
    sentences in episode CSVs: 671  in 72 episode CSV files
    check                    : OK - every sentence is in an episode CSV

  NN_Italy.txt (it)  ->  /content/output/nonnatives/NN_Italy/
    sentences in file        : 554
    matched in .tok          : 554
    not found in .tok        : 0
    duplicates in file       : 0
    sentences in episode C

## 10. Sentences per episode

In [23]:
for r in results:
    print(f"=== {r['corpus']} / {r['folder']} ===")
    display(r["summary"])

=== nonnatives / NN_France ===


,session_id,file_line_first,file_line_last,matched_sentences,speaker_turns,speakers,csv_file
0,ep-01-10-02,1,6,6,1,Vatanen,ep-01-10-02.csv
1,ep-02-01-16,7,9,3,1,Cohn-Bendit,ep-02-01-16.csv
2,ep-02-02-27,10,11,2,1,Cohn-Bendit,ep-02-02-27.csv
3,ep-03-09-02,12,17,6,1,Vatanen,ep-03-09-02.csv
4,ep-03-11-05,18,21,4,1,Cohn-Bendit,ep-03-11-05.csv
5,ep-04-01-14,22,27,6,4,Roche,ep-04-01-14.csv
6,ep-04-01-28,28,90,63,3,Roche,ep-04-01-28.csv
7,ep-04-02-10,91,100,10,1,Roche,ep-04-02-10.csv
8,ep-04-03-10,101,517,417,30,Roche,ep-04-03-10.csv
9,ep-04-03-30,518,632,115,3,Roche,ep-04-03-30.csv


=== nonnatives / NN_Germany ===


,session_id,file_line_first,file_line_last,matched_sentences,speaker_turns,speakers,csv_file
0,ep-01-01-17,1,1,1,1,Jarzembowski,ep-01-01-17.csv
1,ep-01-02-15,2,5,4,1,"Schröder, Ilka",ep-01-02-15.csv
2,ep-01-12-13,6,7,2,2,Pack,ep-01-12-13.csv
3,ep-02-01-16,8,13,6,1,Roth-Behrendt,ep-02-01-16.csv
4,ep-03-06-02,14,35,22,1,"Mann, Erika",ep-03-06-02.csv
...,...,...,...,...,...,...,...
67,ep-98-06-17,657,658,2,1,von Habsburg,ep-98-06-17.csv
68,ep-98-09-14,659,659,1,1,Bangemann,ep-98-09-14.csv
69,ep-98-09-15,660,661,2,1,Bangemann,ep-98-09-15.csv
70,ep-99-01-12,662,664,3,1,von Habsburg,ep-99-01-12.csv


=== nonnatives / NN_Italy ===


,session_id,file_line_first,file_line_last,matched_sentences,speaker_turns,speakers,csv_file
0,ep-01-01-18,1,3,3,1,"Costa, Paolo",ep-01-01-18.csv
1,ep-01-05-02,4,11,8,1,Manisco,ep-01-05-02.csv
2,ep-01-06-12,12,17,6,1,Frassoni,ep-01-06-12.csv
3,ep-01-07-05,18,26,9,1,Manisco,ep-01-07-05.csv
4,ep-01-09-05,27,27,1,1,Frassoni,ep-01-09-05.csv
5,ep-01-11-14,28,30,3,1,Fatuzzo,ep-01-11-14.csv
6,ep-02-09-04,31,40,10,1,Manisco,ep-02-09-04.csv
7,ep-02-09-25,41,47,7,1,Manisco,ep-02-09-25.csv
8,ep-02-11-06,48,63,16,1,Gawronski,ep-02-11-06.csv
9,ep-03-01-29,64,74,11,1,Manisco,ep-03-01-29.csv


=== nonnatives / NN_Spain ===


,session_id,file_line_first,file_line_last,matched_sentences,speaker_turns,speakers,csv_file
0,ep-00-01-19,1,8,8,1,Gorostiaga Atxalandabaso,ep-00-01-19.csv
1,ep-00-02-17,9,14,6,1,Gorostiaga Atxalandabaso,ep-00-02-17.csv
2,ep-00-03-01,15,25,11,1,Gorostiaga Atxalandabaso,ep-00-03-01.csv
3,ep-00-03-16,26,29,4,1,Gorostiaga Atxalandabaso,ep-00-03-16.csv
4,ep-00-04-13,30,36,7,1,Gorostiaga Atxalandabaso,ep-00-04-13.csv
...,...,...,...,...,...,...,...
154,ep-99-10-06,1723,1733,11,2,Gorostiaga Atxalandabaso,ep-99-10-06.csv
155,ep-99-10-27,1734,1741,8,1,Gorostiaga Atxalandabaso,ep-99-10-27.csv
156,ep-99-12-01,1742,1751,10,2,"Barón Crespo, Gorostiaga Atxalandabaso",ep-99-12-01.csv
157,ep-99-12-02,1752,1757,6,1,Gorostiaga Atxalandabaso,ep-99-12-02.csv


## 11. Speakers per sentence file

In [24]:
for r in results:
    if r["speakers"].empty:
        continue
    t, s = r["turns"], r["speakers"]
    print(f"=== {r['corpus']} / {r['folder']} ({r['source_language']}): {int((~s['joint_entry']).sum())} speakers "
          f"(+ {int(s['joint_entry'].sum())} joint entries), {t['session_id'].nunique()} episodes, "
          f"{len(t):,} turns, {int(t['num_lines'].sum()):,} sentences ===")
    display(s.drop(columns=["name_variants", "episodes"]).head(30))

=== nonnatives / NN_France (fr): 12 speakers (+ 0 joint entries), 31 episodes, 77 turns, 1,149 sentences ===


,speaker,joint_entry,mep_id,source_language,num_episodes,num_turns,num_sentences
0,Roche,False,97022,fr,9,54,1012
1,Ari Vatanen,False,4256,fr,8,9,109
2,Cohn-Bendit,False,1934,fr,4,4,11
3,Alain Lipietz,False,4384,fr,2,2,5
4,Jean-Luc Mélenchon,False,96742,fr,1,1,4
5,Pascal Canfin,False,96711,fr,1,1,2
6,Alain Lamassoure,False,1204,fr,1,1,1
7,Bruno Gollnisch,False,1164,fr,1,1,1
8,Catherine Trautmann,False,1129,fr,1,1,1
9,Karima Delli,False,96868,fr,1,1,1


=== nonnatives / NN_Germany (de): 35 speakers (+ 0 joint entries), 72 episodes, 81 turns, 671 sentences ===


,speaker,joint_entry,mep_id,source_language,num_episodes,num_turns,num_sentences
0,Johannes Hahn,False,1636,de,9,12,227
1,Erika Mann,False,1898,de,7,8,79
2,Franziska Katharina Brantner,False,96662,de,6,6,72
3,Alexander Alvaro,False,28246,de,4,4,51
4,von Habsburg,False,1265,de,8,8,25
5,Sven Giegold,False,96730,de,2,2,22
6,Bangemann,False,933,de,4,5,21
7,Tobias Pflüger,False,28249,de,3,3,21
8,Cem Özdemir,False,28235,de,2,2,20
9,Sabine Lösing,False,96854,de,3,3,15


=== nonnatives / NN_Italy (it): 15 speakers (+ 0 joint entries), 57 episodes, 79 turns, 554 sentences ===


,speaker,joint_entry,mep_id,source_language,num_episodes,num_turns,num_sentences
0,Linda McAvan,False,28368,it,13,13,126
1,Manisco,False,2074,it,15,16,124
2,Bonino,False,1566,it,6,21,113
3,Pino Arlacchi,False,97228,it,5,11,66
4,Frassoni,False,4254,it,5,5,48
5,Jas Gawronski,False,1113,it,4,4,25
6,Antonio Tajani,False,2187,it,1,1,14
7,La Malfa,False,1010,it,1,1,11
8,Lilli Gruber,False,28368,it,1,1,11
9,Raffaele Baldassarre,False,96757,it,1,1,6


=== nonnatives / NN_Spain (es): 13 speakers (+ 0 joint entries), 159 episodes, 468 turns, 1,761 sentences ===


,speaker,joint_entry,mep_id,source_language,num_episodes,num_turns,num_sentences
0,Raül Romeva i Rueda,False,28409,es,50,336,934
1,Gorostiaga Atxalandabaso,False,4379,es,73,88,494
2,Bernat Joan i Marí,False,28412,es,18,18,149
3,Ignasi Guardans Cambó,False,28411,es,10,10,93
4,Alejo Vidal-Quadras,False,4308,es,2,2,32
5,Ramon Tremosa i Balcells,False,97203,es,2,2,17
6,Barón Crespo,False,1352,es,3,3,13
7,Vallvé,False,1976,es,3,3,13
8,Willy Meyer,False,28407,es,2,2,9
9,Colom i Naval,False,1349,es,1,1,2


### Example: one speaker, episode by episode

In [25]:
SHOW_SPEAKER = None     # e.g. "Roche"; None -> the speaker with most sentences in the first file

if results and not results[0]["speakers"].empty:
    name = SHOW_SPEAKER or results[0]["speakers"]["speaker"].iloc[0]
    for r in results:
        rows = r["speakers_by_episode"]
        if not rows.empty and (rows["speaker"] == name).any():
            print(f"{name} in {r['corpus']} / {r['folder']}:")
            display(rows[rows["speaker"] == name])

Roche in nonnatives / NN_France:


,speaker,session_id,num_turns,num_sentences
0,Roche,ep-04-01-14,4,6
1,Roche,ep-04-01-28,3,63
2,Roche,ep-04-02-10,1,10
3,Roche,ep-04-03-10,30,417
4,Roche,ep-04-03-30,3,115
5,Roche,ep-04-03-31,7,168
6,Roche,ep-04-04-20,1,31
7,Roche,ep-04-04-21,2,65
8,Roche,ep-04-05-04,3,137


## 12. Overall summary

Written to `output/_overall_summary.csv` (one row per sentence file plus a TOTAL row per corpus)
and `output/_all_speakers.csv` (every speaker of every sentence file in one table).

In [26]:
overall, all_speakers = [], []
for r in results:
    t, s = r["turns"], r["speakers"]
    overall.append({
        "corpus": r["corpus"],
        "file": r["folder"],
        "source_language": r["source_language"],
        "sentences_in_file": r["n_sentences"],
        "not_found_in_tok": r["n_not_found"],
        "num_sentences": int(t["num_lines"].sum()) if not t.empty else 0,
        "num_episodes": t["session_id"].nunique() if not t.empty else 0,
        "num_turns": len(t),
        "distinct_speakers": int((~s["joint_entry"]).sum()) if not s.empty else 0,
        "joint_entries": int(s["joint_entry"].sum()) if not s.empty else 0,
    })
    if not s.empty:
        all_speakers.append(s.assign(corpus=r["corpus"], file=r["folder"]))

overall_df = pd.DataFrame(overall)
rows = []
for corpus_name, part in (overall_df.groupby("corpus", sort=False) if not overall_df.empty else []):
    rows.append(part)
    total = part.drop(columns=["corpus", "file", "source_language"]).sum()
    # an episode can occur in several files, so count distinct episodes
    total["num_episodes"] = pd.concat([r["turns"]["session_id"] for r in results
                                       if r["corpus"] == corpus_name and not r["turns"].empty]).nunique()
    rows.append(pd.DataFrame([{"corpus": corpus_name, "file": "TOTAL", "source_language": "", **total.to_dict()}]))
overall_df = pd.concat(rows, ignore_index=True) if rows else overall_df

all_speakers_df = pd.concat(all_speakers, ignore_index=True) if all_speakers else pd.DataFrame()
if not all_speakers_df.empty:
    front = ["corpus", "file"]
    all_speakers_df = all_speakers_df[front + [c for c in all_speakers_df.columns if c not in front]]

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
save(overall_df, OUTPUT_DIR / "_overall_summary.csv")
save(all_speakers_df, OUTPUT_DIR / "_all_speakers.csv")
display(overall_df)

,corpus,file,source_language,sentences_in_file,not_found_in_tok,num_sentences,num_episodes,num_turns,distinct_speakers,joint_entries
0,nonnatives,NN_France,fr,1149,0,1149,31,77,12,0
1,nonnatives,NN_Germany,de,671,0,671,72,81,35,0
2,nonnatives,NN_Italy,it,554,0,554,57,79,15,0
3,nonnatives,NN_Spain,es,1761,0,1761,159,468,13,0
4,nonnatives,TOTAL,,4135,0,4135,279,705,75,0


In [32]:
!zip -r /content/output.zip /content/output/

updating: content/output/ (stored 0%)
updating: content/output/_overall_summary.csv (deflated 39%)
updating: content/output/nonnatives/ (stored 0%)
updating: content/output/nonnatives/NN_France/ (stored 0%)
updating: content/output/nonnatives/NN_France/ep-99-09-16.csv (deflated 38%)
updating: content/output/nonnatives/NN_France/ep-10-12-15-016.csv (deflated 35%)
updating: content/output/nonnatives/NN_France/_speakers_NN_France.csv (deflated 56%)
updating: content/output/nonnatives/NN_France/ep-06-01-19.csv (deflated 48%)
updating: content/output/nonnatives/NN_France/_summary_NN_France.csv (deflated 62%)
updating: content/output/nonnatives/NN_France/ep-04-03-10.csv (deflated 67%)
updating: content/output/nonnatives/NN_France/ep-06-05-16.csv (deflated 47%)
updating: content/output/nonnatives/NN_France/_speakers_by_episode_NN_France.csv (deflated 55%)
updating: content/output/nonnatives/NN_France/ep-04-01-28.csv (deflated 59%)
updating: content/output/nonnatives/NN_France/ep-03-11-05.csv 